# Topic: Supply Chain Risk Digital Twin
**Students:**  
- Bekithemba Nkomo  
- Masheia Dzimba
- Peter Mangoro

The goal is to represent suppliers, products, and supplier geography as a knowledge graph and then use Cypher and Neo4j Graph Data Science (GDS) to identify high-risk suppliers, risk-concentrated products (single points of failure), and risk clusters by geography.

The desired outcome is an interpretable, network-based view of supply chain vulnerability that supports scenario-style questions such as:

- Which suppliers/products should be prioritized for mitigation?
- Where are systemic risks concentrated?

The solution will be implemented in Neo4j (Desktop), queried in Cypher, and analyzed with at least one GDS workflow in a Jupyter Notebook deliverable.

---
## Impact

- **Operational impact:** Helps prioritize supplier risk mitigation (e.g., diversification, audits, buffer inventory) by highlighting supplier/product nodes that concentrate high delay/disruption risk.
- **Business impact:** Supports better sourcing decisions and reduced disruption costs by identifying "critical" suppliers (network centrality) and high-risk supplier communities.
- **Analytical impact:** Demonstrates how graph modeling improves explainability for supply chain risk compared to purely tabular views (relationships are the signal).

---
## Project Goals

- **Project goal:** Create a supplier–product–country graph model and produce an analytic notebook with interpretable risk insights (rankings, segments/clusters, and vulnerability patterns).
- **Learning goal:** Demonstrate competence in building a knowledge graph in Neo4j and answering graph EDA questions using Cypher.

---
## Literature Review / Environmental Scan / Market Research

- [Kaggle – Supply Chain Dataset (natasha0786)](https://www.kaggle.com/datasets/natasha0786/supply-chain-dataset/data)
- [Gurobi Optimisation Industry Solution Sheet – Supply Chain](https://cdn.gurobi.com/wp-content/uploads/IndustrySolutionSheet-SupplyChainMay2025-1.pdf?x81293)
- [Optimizing Supply Chain Decisions with Pyomo](https://medium.com/@brunoflombardi/optimizing-supply-chain-decisions-with-pyomo-a65b1c620ff4)

---
## Datasets

- **Source:** [Kaggle – Supply Chain Dataset (natasha0786)](https://www.kaggle.com/datasets/natasha0786/supply-chain-dataset/data)

**Notes:**
- ~113k rows
- Contains stable identifiers: `supplier_id`, `product_id` (note: no explicit `supplier_country` field; location is provided as GPS latitude/longitude coordinates)
- Contains risk/performance metrics: `delay_probability`, `disruption_likelihood_score`, `risk_classification`, `lead_time_days`, `shipping_costs`, `delivery_time_deviation`, etc.

---
## Design Concepts

The design is a knowledge graph that models how suppliers connect to products and how supplier geography relates to risk outcomes.

Rather than analyzing each row as an isolated "instance," the graph emphasizes shared suppliers and shared products, enabling network-style insights such as:

- Dependency concentration (many products tied to a few suppliers)
- Risk clustering (groups of suppliers with similar risk profiles and shared product coverage)

The output will include Cypher-based EDA, deeper analytical queries, and at least one GDS-based ranking or clustering result with narrative interpretation.

---
## Conceptual Architecture

### Services/tools

- Neo4j Desktop (local DBMS)
- APOC (schema exploration convenience)
- Neo4j Graph Data Science (GDS)
- Jupyter Notebook + Python driver for reproducible execution and charts

### Data + database considerations

- Create uniqueness constraints for `:Supplier(id)` and `:Product(id)` to ensure clean MERGE ingestion.
- Store row-level risk metrics primarily on relationships `(:Supplier)-[:SUPPLIES]->(:Product)` to represent the risk/performance of that supplier–product pairing.
- Create `:Region(name)` nodes derived from reverse-geocoded GPS coordinates and connect suppliers via `(:Supplier)-[:LOCATED_IN]->(:Region)` for geography-based analysis (no explicit country field exists; all locations are within Southern California).

### Processing/server architecture

- Local single-node Neo4j instance is sufficient (dataset size is moderate).
- Use batching in LOAD CSV to avoid memory spikes.

---
## Analytical Design

### Metrics and signals

- **Supplier risk:** mean/median `delay_probability`, mean `disruption_likelihood_score`, distribution of `risk_classification`
- **Product exposure:** number of distinct suppliers per product (dependency diversification) and risk-weighted exposure
- **Geography:** aggregate risk by reverse-geocoded region/city (derived from GPS lat/lon coordinates; no explicit country field exists in the dataset)
- **Network criticality:** centrality scores (e.g., degree/PageRank/betweenness) in projections such as Supplier–Product or Supplier–Supplier similarity

### Audience

- Supply chain analysts / operations leadership (risk mitigation planning)
- Data/analytics stakeholders evaluating where to invest in resilience improvements

### Outputs and display

- Ranked lists (top risky suppliers, most exposed products, highest-risk countries)
- Distribution plots (risk classification mix by country; supplier reliability vs delay probability)
- GDS outputs (top nodes by centrality; communities with risk profiles)

---
## Sample Question Set

### EDA

- Counts of nodes/relationships after ingestion; distinct labels and rel types
- How many suppliers, products, countries?
- Top 10 suppliers by number of products supplied
- Risk classification distribution overall and by region (derived from GPS coordinates)
- Summary stats of `lead_time_days`, `shipping_costs`, `delay_probability`
- Products with the fewest suppliers (single-sourcing risk)
- Regions with highest average disruption likelihood
- Correlation-style inspection: supplier reliability vs delay probability (bucketed)

### Deeper Cypher 

- Identify **"critical suppliers":** suppliers that supply many products AND have high delay/disruption metrics
- Identify **"high-exposure products":** products whose supplier set is small and/or skewed toward high-risk suppliers
- **Regional vulnerability:** regions where high-risk suppliers dominate critical products
- **Risk concentration:** find product groups that share many of the same suppliers (dependency clusters)

### GDS 

- **PageRank or Degree** on a projected Supplier–Product graph (or Supplier–Supplier co-supply graph) using `delay_probability` or `shipping_costs` as weights, to rank the most "influential/critical" suppliers in the dependency network.
- **Louvain community detection** to find clusters of suppliers/products; compare community risk profiles.

---
## Data Exchange/Processing Framework

### Processing steps

1. Validate CSV schema and data types (numeric parsing, categorical fields).
2. Ingest into Neo4j using LOAD CSV WITH HEADERS:
   - MERGE suppliers, products, countries
   - MERGE/CREATE SUPPLIES relationships with risk/performance properties
3. Run EDA queries to verify schema correctness and scale.
4. Run deeper Cypher questions and capture outputs.
5. Run at least one GDS workflow (estimate → run → interpret), write results back (optional), and report findings.

### Data flow

`CSV → Neo4j (constraints + ingestion) → Cypher EDA/analytics → GDS projection/algorithm → Notebook tables/plots + interpretation`

---
## Additional Comments

The dataset is "instance-based," so the graph model will treat each row as evidence of a supplier–product relationship with associated risk/performance properties.

Where multiple rows exist per supplier–product pair, relationship properties will be aggregated (e.g., mean delay probability, mean disruption likelihood, count of observations).

---
### Feedback Response 1 – GIS Visualizations

The project will incorporate GIS visualizations to provide readers with a clear spatial picture of the supply chain network. Since the dataset provides GPS latitude and longitude for each record, supplier locations will be plotted on an interactive or static map (e.g., using Python's Folium or GeoPandas libraries). Markers will be color-coded by risk classification or delay probability, enabling visual identification of high-risk geographic clusters within the Southern California logistics network.

---
### Feedback Response 2 – Derived Metric Definitions

The dataset contains several derived/composite metrics whose source data and mathematical derivation are not immediately documented. As part of the Final Project deliverable, the team will research and provide a written explanation for each of the following metrics — including the underlying data inputs and the calculation formula or methodology used to produce the values in the dataset:

1. **Supplier Reliability Score**
2. **Route Risk Level**
3. **Driver Behavior Score**
4. **Fatigue Monitoring Score**
5. **Disruption Likelihood Score**
6. **Delay Probability**
7. **Risk Classification**

This explanatory section will appear as a dedicated subsection of the Final Project notebook.

---
### Feedback Response 3 – Geography Scope Revision

The dataset does not contain an explicit country field; location information is limited to GPS latitude and longitude coordinates. The dataset description states the data was collected from a logistics network in **Southern California**, meaning all locations fall within the United States.

Accordingly, the original proposal's reference to aggregating risk "by `supplier_country`" and creating `:Country` nodes has been revised. Geographic analysis will instead use **reverse-geocoded lat/lon coordinates** (e.g., via Python's `geopy` library) to derive city or sub-region groupings within Southern California. If no substantive geographic variation is found at that level, the geographic dimension will be scoped down and the GIS map will serve as the primary geographic deliverable. The GDS community-detection component will be refocused on **risk-profile similarity** rather than geographic proximity if the geographic signal proves insufficient.


---
## References and Hyperlinks

- [Neo4j Graph Data Science docs](https://neo4j.com/docs/graph-data-science/current/)
- [Neo4j Cypher manual](https://neo4j.com/docs/cypher-manual/current/)
- [Kaggle – Supply chain dataset](https://www.kaggle.com/datasets/natasha0786/supply-chain-dataset/data)
- [Barabási, A.-L. – Network Science (open book)](http://networksciencebook.com/)
- [Gurobi Optimisation Industry Solution Sheet](https://cdn.gurobi.com/wp-content/uploads/IndustrySolutionSheet-SupplyChainMay2025-1.pdf?x81293)
- [Optimizing Supply Chain Decisions with Pyomo](https://medium.com/@brunoflombardi/optimizing-supply-chain-decisions-with-pyomo-a65b1c620ff4)

